# Evaluación de LLMs comerciales: ChatGPT vs Claude

Cuaderno para medir ChatGPT y Claude sobre resúmenes médicos en lenguaje sencillo: reproduce el flujo del Colab original, separa entornos (Py3.12 + venv Py3.10 para AlignScore) y deja registro de métricas y artefactos.

## Tabla de contenidos

- [0. Expectativas de entorno](#0-expectativas-de-entorno)

- [1. Verificación de runtime](#1-verificacion-de-runtime)

- [2. Instalación de dependencias](#2-instalacion-de-dependencias)

- [3. Carga de claves y configuración compartida](#3-carga-de-claves-y-configuracion-compartida)

- [4. Configuración central de evaluación](#4-configuracion-central-de-evaluacion)

- [5. Carga del dataset](#5-carga-del-dataset)

- [6. Clientes de OpenAI y Anthropic](#6-clientes-de-openai-y-anthropic)

- [7. Formateo de prompts](#7-formateo-de-prompts)

- [8. Generación por proveedor](#8-generacion-por-proveedor)

- [9. Orquestador de inferencia en lote](#9-orquestador-de-inferencia-en-lote)

- [10. Utilidades de métricas](#10-utilidades-de-metricas)

- [11. Evaluación de generaciones](#11-evaluacion-de-generaciones)

- [12. Flujo aislado para AlignScore](#12-flujo-aislado-para-alignscore)

- [13. Agregación y comparación](#13-agregacion-y-comparacion)

- [14. Persistencia de salidas](#14-persistencia-de-salidas)


Notebook alineado con `pls_sft_qwen.ipynb`, pero dedicado a evaluar APIs comerciales. El runtime principal es Python 3.12 y AlignScore corre en un venv de Python 3.10 para mantener compatibilidad con Torch 1.13 sin afectar el resto.


## 0. Expectativas de entorno

Trabaja en Python 3.12 para el flujo principal. Ejemplo de creación de entorno:

```bash
python3.12 -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
```

En Colab selecciona el runtime de Python 3.12. Más adelante se levanta un venv en Python 3.10 para AlignScore siguiendo el patrón del cuaderno original, manteniendo el resto del stack en 3.12.


## 1. Verificación de runtime

Comprueba la versión de Python, SO y GPU para decidir si se requiere alguna bifurcación o si se continúa sin aceleración.


In [ ]:
import platform
import sys
import torch

print(f"Python: {sys.version}")
if not sys.version.startswith('3.12'):
    print('⚠️ Recommended interpreter is Python 3.12.x (AlignScore uses its own 3.10 venv later).')
print(f"Platform: {platform.platform()}")
print(f"PyTorch: {torch.__version__}")
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.105+-x86_64-with-glibc2.35
PyTorch: 2.8.0+cu126
CUDA available: False


## 2. Instalación de dependencias

Instala SDKs de OpenAI/Anthropic y librerías de evaluación (BERTScore, métricas de legibilidad, AlignScore) con versiones fijas compatibles tanto con el flujo principal como con el entorno 3.10 aislado.


In [ ]:
%%bash
set -euo pipefail
pip install --upgrade   openai==2.7.2  anthropic==0.72.1   python-dotenv==1.0.1   pandas==2.2.2   textstat==0.7.4   evaluate==0.4.2   bert-score==0.3.13   datasets==2.20.0   tqdm==4.66.4   numpy==2.1.3   torch==2.8.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.4/357.4 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.1/105.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 110.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 49.4 MB/s eta 0:00:00
  Attempting uninstall: tqdm
    Found existing i

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 0.8.3 requires tqdm>=4.67, but you have tqdm 4.66.4 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.5.0 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.1.3 which is incompatible.


## 3. Carga de claves y configuración compartida

Lee credenciales desde `.env` o variables de entorno para mantener los secretos fuera de las salidas del notebook.


In [ ]:
import os
from google.colab import userdata


OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
if not OPENAI_API_KEY:
    print('Missing OPENAI_API_KEY in env/.env')
if not ANTHROPIC_API_KEY:
    print('Missing ANTHROPIC_API_KEY in env/.env')

## 4. Configuración central de evaluación

Centraliza rutas de dataset, prompts, modelos a consultar, límites de tokens y banderas de seguridad en una dataclass fácil de ajustar.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional
import pandas as pd

In [ ]:
#SYS_PLS = (
#  "You are an expert biomedical plain-language writer. "
#  "Write the Plain Language Summary (PLS) in clear, natural English ONLY based on the medical or clinical texts. "
#  "One paragraph only; 4–6 sentences, 12–16 words each; third person, active voice. "
#  "Define unavoidable jargon once (in parentheses). "
#  "No headings, lists, citations, URLs, or source mentions. "
#  "Use only facts from the article; do not add new facts, opinions or medical recomendations. "
#  "Targets: Flesch≥60; FKGL≤6; Fog≤8; SMOG≤8; CLI≤8; Dale–Chall≤8. "
#  "If information is missing, leave it out. Do not mention these instructions."
#)

SYS_PLS = (
  "*[SYSTEM INSTRUCTION]*"
  "You are a Health Literacy Expert. Your expertise is in rewriting complex, technical medical texts into clear, simple, and accurate language for a general audience, following established health communication guidelines. "
  "*[PRIMARY GOAL]*"
  "Your main purpose is to rewrite the provided medical text into a Plain Language Summary (PLS). This summary must be easy to understand for someone with an 8th-grade reading comprehension level typical of a general middle-school student, consistent with plain-language standards used by CDC and NIH, meaning it should use simple vocabulary, short sentences, and concepts that can be understood by someone with basic middle-school literacy, while remaining completely faithful to the source's essential information, for intance, conclusions. "
  "*[TASK INSTRUCTION]* "
  "Rewrite the following technical medical text into a Plain Language Summary (PLS). The output *MUST BE ONLY* the plain language summary without special symbols and stop tokens. "
  "*--- STRICT OUTPUT RULES ---* "
  "*1.  *ACCURACY AND COMPLETENESS:* "
  "*   The summary MUST retain all key findings, main outcomes, important safety information, conclusions, and any significant numerical results from the original text. "
  "*   Do NOT add any information, opinions, or recommendations that are not present in the source document. The summary must be based ONLY on the provided text. "
  "*2.  *CLARITY AND READABILITY:* "
  "*   Write the summary at an *8th-grade reading comprehension level typical of a general middle-school student*. "
  "*   Use short, clear, and natural-sounding sentences. "
  "*   Use the active voice whenever possible (e.g., \"Scientists tested the drug\" instead of \"The drug was tested by scientists\"). "
  "*   Avoid long, complex words when a simpler alternative exists. "
  "*3.  *TERMINOLOGY (JARGON):* "
  "*   Avoid medical jargon. "
  "*   If a technical term is absolutely essential and cannot be replaced, you MUST explain it simply in parentheses the first time it appears. (Example: \"The trial used immunotherapy (a treatment that helps the body's immune system fight cancer).\") "
  "*4.  *FORMATTING AND LANGUAGE:* "
  "*   The output must be written *ONLY in English*. "
  "*   Structure the summary as a single, concise paragraph. Use as many sentences as needed to include all essential information, up to a maximum length of about 7 sentences, however, *make sure ALL sentences are complete, which means ALL ideas are finished.* "
  "*   Do NOT include headings, bullet points, lists, citations, or URLs. "
  "*--- SOURCE TECHNICAL TEXT ---* "
)

SYS_PLS = (
  "*[SYSTEM INSTRUCTION]*"
  "You are a Health Literacy Expert. Your expertise is in rewriting complex, technical medical texts into clear, simple, and accurate language for a general audience, following established health communication guidelines. "
  "*[PRIMARY GOAL]*"
  "Your main purpose is to rewrite the provided medical text into a Plain Language Summary (PLS). This summary must be easy to understand for someone with an 8th-grade reading comprehension level typical of a general middle-school student, consistent with plain-language standards used by CDC and NIH, meaning it should use simple vocabulary, short sentences, and concepts that can be understood by someone with basic middle-school literacy, while remaining completely faithful to the source's essential information, for intance, conclusions. "
)

USER_PLS = (
  "*[TASK INSTRUCTION]* "
  "Rewrite the following technical medical text into a Plain Language Summary (PLS). The output *MUST BE ONLY* the plain language summary without special symbols and stop tokens. "
  "*--- STRICT OUTPUT RULES ---* "
  "*1.  *ACCURACY AND COMPLETENESS:* "
  "*   The summary MUST retain all key findings, main outcomes, important safety information, conclusions, and any significant numerical results from the original text. "
  "*   Do NOT add any information, opinions, or recommendations that are not present in the source document. The summary must be based ONLY on the provided text. "
  "*2.  *CLARITY AND READABILITY:* "
  "*   Write the summary at an *8th-grade reading comprehension level typical of a general middle-school student*. "
  "*   Use short, clear, and natural-sounding sentences. "
  "*   Use the active voice whenever possible (e.g., \"Scientists tested the drug\" instead of \"The drug was tested by scientists\"). "
  "*   Avoid long, complex words when a simpler alternative exists. "
  "*3.  *TERMINOLOGY (JARGON):* "
  "*   Avoid medical jargon. "
  "*   If a technical term is absolutely essential and cannot be replaced, you MUST explain it simply in parentheses the first time it appears. (Example: \"The trial used immunotherapy (a treatment that helps the body's immune system fight cancer).\") "
  "*4.  *FORMATTING AND LANGUAGE:* "
  "⁠*   The output *MUST BE ONLY* the plain language summary without special symbols and stop tokens, *only the summary*."
  "*   The output must be written *ONLY in English*. "
  "⁠*   Structure the summary as a set of concise paragraphs. Use as many sentences as needed to include all essential information, up to a maximum length of about 500 words, however, *make sure ALL sentences are complete, which means ALL ideas are finished.*"
  "*   Do NOT include headings, bullet points, lists, citations, or URLs. "
  "*--- SOURCE TECHNICAL TEXT ---* "
  "\n⁠<document>"
	"⁠{technical_text}"
	"⁠</document>"

)

SYS_PLS_V2 = (
  "You are a biomedical communication specialist trained to create factually precise Plain Language Summaries (PLS). "
  "Rewrite the medical or clinical text below in one coherent paragraph, 4–6 sentences long. "
  "Each sentence should use 12–16 words, active voice, and third person. "
  "Keep every verified medical fact, relationship, and quantitative result exactly as stated. "
  "Simplify only the language—not the meaning—using clear, everyday English. "
  "Define unavoidable technical terms once in parentheses and avoid speculation or advice. "
  "Do not copy phrases; paraphrase faithfully to preserve the same meaning (improves semantic alignment). "
  "Ensure the summary would score high for factual consistency (AlignScore) and semantic similarity (BERTScore) "
  "Ensure Also this Targets to warrant readability: Flesch≥60; FKGL≤6; Fog≤8; SMOG≤8; CLI≤8; Dale–Chall≤8. "
  "compared with the source. "
  "Avoid headings, lists, citations, URLs, or mentions of the original source. "
  "Write naturally; do not mention these instructions."
)

SYS_PLS_V3 = (
  "You are a biomedical plain-language specialist. Rewrite the source text as ONE"
  " paragraph (no headings), 4–6 sentences, each 10–16 words, in active voice and"
  " third person. Use clear everyday English and short, simple sentence structure"
  " (no semicolons, no nested clauses, minimal commas). Replace jargon with common"
  " words; define unavoidable terms once in parentheses. "
  "Preserve every key medical fact and relationship from the source. Do not omit"
  " the study purpose, what was compared, the main effects (direction and rough"
  " size: small/moderate/none), the confidence/limitations (evidence quality,"
  " heterogeneity), and any safety findings. Do not copy phrases; paraphrase while"
  " keeping the same meaning to maximize semantic alignment (BERTScore) and factual"
  " consistency (AlignScore). Avoid numbers and statistics unless central to the"
  " claim; if used, state them simply. Do not add opinions, advice, or new facts."
  " Do not include lists, citations, URLs, or mention the source or these rules."
)

SYS_PLS_V4 = (
  "You are a biomedical communication specialist trained to create precise and trustworthy Plain Language Summaries (PLS)."
  "Rewrite the medical or clinical text below into one coherent paragraph of 4–6 sentences."
  "Use 12–16 words per sentence, active voice, and third person."
  "Keep every medical fact, relationship, and numerical result exactly as stated in the source."
  "Do not add, infer, or remove information."
  "Simplify the language only—not the meaning—using clear, everyday English ONLY."
  "Explain unavoidable technical terms once in parentheses."
  "Paraphrase all wording to avoid copying phrases from the original text."
  "Aim for strong factual consistency (AlignScore) and semantic similarity (BERTScore)."
  "Aim for high readability: Flesch ≥ 60; FKGL ≤ 6; Fog ≤ 8; SMOG ≤ 8; CLI ≤ 8; Dale–Chall ≤ 8."
  "Avoid headings, lists, citations, URLs, or references to the original text."
  "Write naturally without mentioning these instructions."
)

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional


def default_align_env_path() -> Path:
    colab_root = Path('/content')
    if colab_root.exists():
        return colab_root / 'envs' / 'py310-alignscore'
    return Path.home() / '.venvs' / 'py310-alignscore'


@dataclass
class EvalConfig:
    data_path: Path = Path('/content/drive/MyDrive/pls_agent/data/v2/evaluation_texts.csv')
    #data_path: Path = Path('/content/drive/MyDrive/pls_agent/results/qwen_pls.csv')
    output_path: Path = Path('/content/drive/MyDrive/pls_agent/results/commercial_llm_eval_team_v2.csv')
    #output_path: Path = Path('/content/drive/MyDrive/pls_agent/results/qwen_llm_eval_team.csv')
    system_prompt: str = SYS_PLS
    user_template: str = USER_PLS
    openai_model: str = 'gpt-5.1'
    #anthropic_model: str = 'claude-3-haiku-20240307'
    anthropic_model: str = 'claude-sonnet-4-5-20250929'
    temperature: float = 0.2
    max_output_tokens: int = 512
    max_requests: Optional[int] = 100
    dry_run: bool = False  # flip to True to inspect the pipeline without hitting APIs
    alignscore_input_path: Path = Path('../results/qwen_alignscore_payload_team.csv')
    alignscore_output_path: Path = Path('../results/qwen_alignscore_scores_team.csv')
    alignscore_env_path: Path = field(default_factory=default_align_env_path)


config = EvalConfig()
config


EvalConfig(data_path=PosixPath('/content/drive/MyDrive/pls_agent/data/v2/evaluation_texts.csv'), output_path=PosixPath('/content/drive/MyDrive/pls_agent/results/commercial_llm_eval_team_v2.csv'), system_prompt="*[SYSTEM INSTRUCTION]*You are a Health Literacy Expert. Your expertise is in rewriting complex, technical medical texts into clear, simple, and accurate language for a general audience, following established health communication guidelines. *[PRIMARY GOAL]*Your main purpose is to rewrite the provided medical text into a Plain Language Summary (PLS). This summary must be easy to understand for someone with an 8th-grade reading comprehension level typical of a general middle-school student, consistent with plain-language standards used by CDC and NIH, meaning it should use simple vocabulary, short sentences, and concepts that can be understood by someone with basic middle-school literacy, while remaining completely faithful to the source's essential information, for intance, concl

## 5. Carga del dataset

Lee pares artículo/referencia en un DataFrame y permite recortar con `config.max_requests` para pruebas rápidas de costo bajo.


In [ ]:
import pandas as pd

if not config.data_path.exists():
    raise FileNotFoundError(f'Dataset not found: {config.data_path}')
raw_df = pd.read_csv(config.data_path)
required_cols = {'technical_text', 'reference_summary'}
missing = required_cols.difference(raw_df.columns)
if missing:
    raise ValueError(f'Dataset is missing columns: {missing}')
if config.max_requests:
    raw_df = raw_df.head(config.max_requests)
print(raw_df.head(2))
print(f'Total samples: {len(raw_df)}')

                                      technical_text  \
0  Background\nLumbar spinal stenosis with neurog...   
1  Background\nOlder patients with multiple healt...   

                                   reference_summary  
0  Non‐surgical treatment for spinal stenosis wit...  
1  Interventions for involving older patients wit...  
Total samples: 100


## 6. Clientes de OpenAI y Anthropic

Inicializa los SDKs bajo demanda para que el notebook pueda ejecutarse en modo analítico aun sin credenciales cargadas.


In [ ]:
from typing import Optional
from openai import OpenAI
from anthropic import Anthropic

openai_client: Optional[OpenAI] = None
anthropic_client: Optional[Anthropic] = None

if OPENAI_API_KEY:
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
if ANTHROPIC_API_KEY:
    anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)

print('OpenAI client ready:', bool(openai_client))
print('Anthropic client ready:', bool(anthropic_client))

OpenAI client ready: True
Anthropic client ready: True


## 7. Formateo de prompts

Arma el prompt por artículo de forma consistente antes de enviarlo a cualquiera de los proveedores.


In [ ]:
def build_prompt(technical_text: str) -> str:
    return config.user_template.format(technical_text=technical_text.strip())

raw_df = raw_df.assign(prompt=raw_df['technical_text'].apply(build_prompt))
#raw_df.drop(['generated_summary'], axis='columns', inplace=True)
raw_df.head(2)

,technical_text,reference_summary,prompt
0,Background\nLumbar spinal stenosis with neurog...,Non‐surgical treatment for spinal stenosis wit...,*[TASK INSTRUCTION]* Rewrite the following tec...
1,Background\nOlder patients with multiple healt...,Interventions for involving older patients wit...,*[TASK INSTRUCTION]* Rewrite the following tec...


## 8. Generación por proveedor

Envuelve las llamadas específicas a cada API, controla el conteo de tokens y respeta `config.dry_run` para evitar cargos accidentales.


In [ ]:
from time import perf_counter
from typing import Dict, Any

class InferenceError(RuntimeError):
    pass


def _maybe_abort(provider: str):
    if config.dry_run:
        raise InferenceError(f'Dry-run enabled; skipping {provider} call')


def generate_with_chatgpt(prompt: str) -> Dict[str, Any]:
    if openai_client is None:
        raise InferenceError('OpenAI client is not configured')
    _maybe_abort('OpenAI')
    start = perf_counter()
    response = openai_client.responses.create(
        model=config.openai_model,
        input=[
            {"role": "system", "content": config.system_prompt},
            {"role": "user", "content": prompt},
        ],
        temperature=config.temperature,
        max_output_tokens=config.max_output_tokens,
    )
    text_chunks = []
    for item in response.output:
        for content in item.content:
            if content.type == 'output_text':
                text_chunks.append(content.text)
    generation = ''.join(text_chunks).strip()
    usage = response.usage
    elapsed = perf_counter() - start
    return {
        'generation': generation,
        'prompt_tokens': getattr(usage, 'input_tokens', None),
        'completion_tokens': getattr(usage, 'output_tokens', None),
        'total_tokens': getattr(usage, 'total_tokens', None),
        'latency_seconds': elapsed,
    }


def generate_with_claude(prompt: str) -> Dict[str, Any]:
    if anthropic_client is None:
        raise InferenceError('Anthropic client is not configured')
    _maybe_abort('Anthropic')
    start = perf_counter()
    response = anthropic_client.messages.create(
        model=config.anthropic_model,
        system=config.system_prompt,
        temperature=config.temperature,
        max_tokens=config.max_output_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    text_chunks = [block.text for block in response.content if block.type == 'text']
    generation = ''.join(text_chunks).strip()
    usage = response.usage
    elapsed = perf_counter() - start
    return {
        'generation': generation,
        'prompt_tokens': getattr(usage, 'input_tokens', None),
        'completion_tokens': getattr(usage, 'output_tokens', None),
        'total_tokens': getattr(usage, 'input_tokens', 0) + getattr(usage, 'output_tokens', 0),
        'latency_seconds': elapsed,
    }

## 9. Orquestador de inferencia en lote

Itera sobre el dataset, delega en el generador correspondiente y almacena telemetría para las métricas posteriores.


In [ ]:
from tqdm import tqdm

records = []

def run_model(df: pd.DataFrame, provider: str) -> None:
    generator = generate_with_chatgpt if provider == 'chatgpt' else generate_with_claude
    for row in tqdm(df.itertuples(index=False), total=len(df), desc=provider):
        try:
            result = generator(row.prompt)
        except InferenceError as err:
            print(f'Skipping sample due to {err}')
            continue
        except Exception as exc:
            print(f'Provider {provider} failed: {exc}')
            continue
        records.append({
            'provider': provider,
            'technical_text': row.technical_text,
            'reference_summary': row.reference_summary,
            'prompt': row.prompt,
            **result,
        })

for provider in ['chatgpt', 'claude']:
#for provider in ['claude']:
    print(f'Running provider: {provider}')
    run_model(raw_df, provider)

results_df = pd.DataFrame(records).reset_index(drop=True)
results_df['row_id'] = results_df.index
results_df.head()


Running provider: chatgpt


chatgpt: 100%|██████████| 100/100 [15:10<00:00,  9.10s/it]


Running provider: claude


claude: 100%|██████████| 100/100 [22:30<00:00, 13.50s/it]


,provider,technical_text,reference_summary,prompt,generation,prompt_tokens,completion_tokens,total_tokens,latency_seconds,row_id
0,chatgpt,Background\nLumbar spinal stenosis with neurog...,Non‐surgical treatment for spinal stenosis wit...,*[TASK INSTRUCTION]* Rewrite the following tec...,Lumbar spinal stenosis with neurogenic claudic...,1187,512,1699,13.714724,0
1,chatgpt,Background\nOlder patients with multiple healt...,Interventions for involving older patients wit...,*[TASK INSTRUCTION]* Rewrite the following tec...,This review looked at whether certain programs...,1835,512,2347,11.102713,1
2,chatgpt,Background\nBeta‐blockers are an essential par...,Beta‐blockers for children with congestive hea...,*[TASK INSTRUCTION]* Rewrite the following tec...,Beta-blockers are medicines that help treat he...,1143,506,1649,7.897319,2
3,chatgpt,Background\nThalassaemia is a genetic disorder...,Removal of the spleen in people with thalassae...,*[TASK INSTRUCTION]* Rewrite the following tec...,Thalassemia is an inherited blood disease that...,1483,512,1995,10.223576,3
4,chatgpt,Background\nApproximately 600 million children...,"One, two or three times a week iron supplement...",*[TASK INSTRUCTION]* Rewrite the following tec...,Many children around the world do not have eno...,1603,512,2115,6.774514,4


In [ ]:
import pandas as pd
from pathlib import Path

output_csv = Path('/content/drive/MyDrive/pls_agent/results/predictions_commercial_team_v2.csv')
# Ensure the output directory exists
output_csv.parent.mkdir(parents=True, exist_ok=True)

# Save the results_df to the specified path
results_df.to_csv(output_csv)
print(f'Wrote detailed results to {output_csv}')

Wrote detailed results to /content/drive/MyDrive/pls_agent/results/predictions_commercial_team_v2.csv


In [ ]:
len(results_df)

200

## 10. Utilidades de métricas

Define scorers de BERTScore, legibilidad (Flesch, FKGL, Gunning Fog, SMOG, Coleman-Liau) y AlignScore, devolviendo NaN si AlignScore no puede cargarse en el hardware disponible.


In [ ]:
import numpy as np
import pandas as pd
import evaluate
from textstat import textstat

#_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
_DEVICE = 'cpu'
bertscore = evaluate.load('bertscore')




In [ ]:
def compute_bertscore(df: pd.DataFrame) -> pd.Series:
    payload = bertscore.compute(
        predictions=df['generation'].tolist(),
        references=df['reference_summary'].tolist(),
        model_type='microsoft/deberta-large-mnli',
        #model_type='microsoft/deberta-xlarge-mnli',
        lang='en',
        device=_DEVICE,
    )
    return pd.Series(payload['f1'], index=df.index)


def readability_scores(text: str) -> dict:
    return {
        'flesch_reading_ease': textstat.flesch_reading_ease(text),
        'flesch_kincaid_grade': textstat.flesch_kincaid_grade(text),
        'gunning_fog': textstat.gunning_fog(text),
        'smog_index': textstat.smog_index(text),
        'coleman_liau_index': textstat.coleman_liau_index(text),
    }

In [ ]:
from pathlib import Path

#output_csv = Path('/content/drive/MyDrive/pls_agent/results/qwen_pls.csv')
output_csv = Path('/content/drive/MyDrive/pls_agent/results/predictions_commercial_team_v2.csv')
results_df = pd.read_csv(output_csv)

In [ ]:
results_df.head(10)

In [ ]:
results_df = results_df.rename(columns={'qwen_pls': 'generation'})
results_df['provider'] = 'qwen+finetuned'

In [ ]:
len(results_df)

200

## 11. Evaluación de generaciones

Aplica todas las métricas a los grupos por proveedor y agrega los resultados al `results_df`.


In [ ]:
metric_frames = []
for provider, group in results_df.groupby('provider'):
    idx = group.index
    bert = compute_bertscore(group)
    readability = group['generation'].apply(readability_scores).apply(pd.Series)
    metrics = pd.DataFrame({
        'provider': provider,
        'bertscore_f1': bert,
    }, index=idx).join(readability)
    metrics['align_score'] = np.nan
    metric_frames.append(metrics)
metrics_df = pd.concat(metric_frames).sort_index()
results_df = results_df.join(metrics_df.drop(columns=['provider']))
results_df.head()

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

,Unnamed: 0,provider,technical_text,reference_summary,prompt,generation,prompt_tokens,completion_tokens,total_tokens,latency_seconds,row_id,bertscore_f1,flesch_reading_ease,flesch_kincaid_grade,gunning_fog,smog_index,coleman_liau_index,align_score
0,0,chatgpt,Background\nLumbar spinal stenosis with neurog...,Non‐surgical treatment for spinal stenosis wit...,*[TASK INSTRUCTION]* Rewrite the following tec...,Lumbar spinal stenosis with neurogenic claudic...,1187,512,1699,13.714724,0,0.632556,62.27,8.9,9.56,11.7,11.95,NaN
1,1,chatgpt,Background\nOlder patients with multiple healt...,Interventions for involving older patients wit...,*[TASK INSTRUCTION]* Rewrite the following tec...,This review looked at whether certain programs...,1835,512,2347,11.102713,1,0.659624,48.23,12.2,12.38,12.9,13.70,NaN
2,2,chatgpt,Background\nBeta‐blockers are an essential par...,Beta‐blockers for children with congestive hea...,*[TASK INSTRUCTION]* Rewrite the following tec...,Beta-blockers are medicines that help treat he...,1143,506,1649,7.897319,2,0.684504,60.55,9.6,9.83,10.6,14.10,NaN
3,3,chatgpt,Background\nThalassaemia is a genetic disorder...,Removal of the spleen in people with thalassae...,*[TASK INSTRUCTION]* Rewrite the following tec...,Thalassemia is an inherited blood disease that...,1483,512,1995,10.223576,3,0.653123,54.02,10.0,9.72,12.1,13.05,NaN
4,4,chatgpt,Background\nApproximately 600 million children...,"One, two or three times a week iron supplement...",*[TASK INSTRUCTION]* Rewrite the following tec...,Many children around the world do not have eno...,1603,512,2115,6.774514,4,0.662102,68.70,8.5,9.26,10.0,11.02,NaN


In [ ]:
metrics_df

In [ ]:
summary = (
    results_df.groupby('provider')[
        [
            'prompt_tokens',
            'completion_tokens',
            'total_tokens',
            'latency_seconds',
            'bertscore_f1',
            #'align_score',
            'flesch_reading_ease',
            'flesch_kincaid_grade',
            'gunning_fog',
            'smog_index',
            'coleman_liau_index',
        ]
    ]
    .mean()
    .rename(columns={'latency_seconds': 'avg_latency_seconds'})
)
summary

,prompt_tokens,completion_tokens,total_tokens,avg_latency_seconds,bertscore_f1,flesch_reading_ease,flesch_kincaid_grade,gunning_fog,smog_index,coleman_liau_index
provider,,,,,,,,,,
chatgpt,1640.76,502.93,2143.69,9.100427,0.637283,61.3862,8.989,9.2845,11.118,11.8401
claude,1831.94,496.04,2327.98,13.502871,0.633787,57.6027,9.921,10.0396,11.618,13.1558


In [ ]:
output_csv = Path('/content/drive/MyDrive/pls_agent/results/predictions_commercial_metrics_team_v2.csv')
# Ensure the output directory exists
output_csv.parent.mkdir(parents=True, exist_ok=True)

# Save the results_df to the specified path
results_df.to_csv(output_csv)
print(f'Wrote detailed results to {output_csv}')

Wrote detailed results to /content/drive/MyDrive/pls_agent/results/predictions_commercial_metrics_team_v2.csv


## 12. Flujo aislado para AlignScore

Ejecuta AlignScore en un entorno virtual de Python 3.10 con Torch 1.13: exporta muestras generadas, las puntúa en ese venv y vuelve a fusionar los resultados. Puede omitirse si no se requiere AlignScore.


In [ ]:
output_csv = Path('/content/drive/MyDrive/pls_agent/results/predictions_commercial_metrics_team_align_v2.csv')
results_df = pd.read_csv(output_csv)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/pls_agent/results/predictions_commercial_metrics_team_align_v2.csv'

In [ ]:
len(results_df)

In [ ]:
results_df['row_id'] = results_df.index

In [ ]:
align_payload_cols = ['row_id', 'technical_text', 'generation', 'reference_summary']
align_payload = results_df[align_payload_cols].copy()
align_payload_path = config.alignscore_input_path
align_payload_path.parent.mkdir(parents=True, exist_ok=True)
align_payload.to_csv(align_payload_path, index=False)
print(f'Saved {len(align_payload)} rows to {align_payload_path}')
print(f'Default AlignScore venv: {config.alignscore_env_path}')
print('Set ALIGN_VENV / ALIGNSCORE_INPUT / ALIGNSCORE_OUTPUT to override defaults if needed.')


In [ ]:
%%bash
# Create a fresh Python 3.10 env dedicated to AlignScore (safe for Colab)
set -euo pipefail
ENV_DIR=${ALIGN_VENV:-/content/envs/py310-alignscore}

sudo apt-get update -y >/dev/null
sudo apt-get install -y python3.10 python3.10-distutils python3.10-venv >/dev/null
if ! python3.10 -m ensurepip --upgrade >/dev/null 2>&1; then
  curl -sS https://bootstrap.pypa.io/get-pip.py | sudo python3.10
fi

rm -rf "${ENV_DIR}"
python3.10 -m venv "${ENV_DIR}"
source "${ENV_DIR}/bin/activate"

python -m pip install --upgrade pip wheel setuptools
pip install --no-cache-dir torch==1.13.1
pip install --no-cache-dir numpy==1.26.4 pandas==2.2.2 requests==2.32.3
pip install --no-cache-dir transformers==4.40.1 sentencepiece==0.1.99
pip install --no-cache-dir "git+https://github.com/yuh-zha/AlignScore.git"

python - <<'PY'
import sys, torch
print('Python:', sys.executable)
print('Torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
import alignscore
print('AlignScore import OK')
PY


In [ ]:
%%bash
ENV_DIR=${ALIGN_VENV:-/content/envs/py310-alignscore}
source "${ENV_DIR}/bin/activate"

pip install --no-cache-dir spacy
python -m spacy download en_core_web_sm


In [ ]:
%%bash
# Quick sanity check
python - <<'PY'
import spacy
nlp = spacy.load("en_core_web_sm")
print("spaCy model loaded OK in AlignScore env:", nlp.meta.get("name"), nlp.meta.get("version"))
PY

In [ ]:
%%bash
# Run AlignScore inside the dedicated env and write outputs back to CSV
set -euo pipefail
VENV=${ALIGN_VENV:-/content/envs/py310-alignscore}
CSV_IN=${ALIGNSCORE_INPUT:-../results/predictions_commercial_metrics_team_align_v2.csv}
CSV_OUT=${ALIGNSCORE_OUTPUT:-../results/commercial_alignscore_scores_team_v2.csv}

if [ ! -d "${VENV}" ]; then
  echo "AlignScore venv not found at ${VENV}. Run the setup cell first."
  exit 1
fi
if [ ! -f "${CSV_IN}" ]; then
  echo "Payload not found at ${CSV_IN}. Re-run the export cell."
  exit 1
fi

source "${VENV}/bin/activate"
export HF_HUB_ENABLE_HF_TRANSFER=0
export HF_HUB_DISABLE_TELEMETRY=1

python - <<'PY'
import os
from pathlib import Path
import numpy as np
import pandas as pd

import nltk

nltk.download('punkt_tab')

csv_in = Path(os.environ.get('ALIGNSCORE_INPUT', '../results/predictions_commercial_metrics_team_align_v2.csv'))
csv_out = Path(os.environ.get('ALIGNSCORE_OUTPUT', '../results/commercial_alignscore_scores_team_v2.csv'))

if not csv_in.exists():
    raise FileNotFoundError(f'AlignScore payload missing: {csv_in}')

df = pd.read_csv(csv_in)
required = {'row_id', 'technical_text', 'generation'}
missing = required.difference(df.columns)
if missing:
    raise RuntimeError(f'Payload missing columns: {missing}')

def _clean(series):
    return series.fillna('').map(lambda t: ' '.join(str(t).split()).strip())

df['_technical_text'] = _clean(df['technical_text'])
df['_generation'] = _clean(df['generation'])
if 'reference_summary' in df.columns:
    df['_reference'] = _clean(df['reference_summary'])

mask = df['_technical_text'].str.len().ge(30) & df['_generation'].str.len().ge(30)
valid = df.loc[mask].copy()

from alignscore import AlignScore

model_name = os.environ.get('ALIGNSCORE_MODEL', 'roberta-base')
batch_size = int(os.environ.get('ALIGNSCORE_BATCH', '4'))
device = 'cuda' if os.environ.get('CUDA_VISIBLE_DEVICES') else 'cpu'
ckpt_path = os.environ.get(
    'ALIGNSCORE_CKPT',
    os.path.join(os.environ.get('ALIGN_VENV', '/content/drive/MyDrive'),
                 'AlignScore-base.ckpt'),
)

scores = [np.nan] * len(df)
if not valid.empty:
    scorer = AlignScore(model=model_name, batch_size=batch_size, device=device, ckpt_path=ckpt_path)
    references = valid['_reference'].tolist() if '_reference' in valid.columns else None
    scored = scorer.score(
        contexts=valid['_technical_text'].tolist(),
        claims=valid['_generation'].tolist(),
    )
    for idx, value in zip(valid.index, scored):
        scores[idx] = value

out = df[['row_id']].copy()
out['align_score'] = scores
out.to_csv(csv_out, index=False)
print(f'Saved AlignScore scores to {csv_out}')
PY


Saved AlignScore scores to ../results/qwen_alignscore_scores_team.csv


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
/content/envs/py310-alignscore/lib/python3.10/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
/content/envs/py310-alignscore/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should

In [ ]:
align_output = config.alignscore_output_path
if align_output.exists():
    align_df = pd.read_csv(align_output)
    required = {'row_id', 'align_score'}
    if required.issubset(align_df.columns):
        score_map = align_df.drop_duplicates('row_id').set_index('row_id')['align_score']
        results_df['align_score'] = results_df['row_id'].map(score_map)
        print(f'Merged AlignScore values from {align_output}')
    else:
        print(f'AlignScore output missing columns: {align_df.columns}')
else:
    print(f'AlignScore output not found ({align_output}); align_score remains NaN')
results_df[['provider', 'row_id', 'align_score']].head()


Merged AlignScore values from ../results/qwen_alignscore_scores_team.csv


,provider,row_id,align_score
0,qwen+finetuned,0,0.312384
1,qwen+finetuned,1,0.257904
2,qwen+finetuned,2,0.572171
3,qwen+finetuned,3,0.222919
4,qwen+finetuned,4,0.488934


In [ ]:
align_df

,row_id,align_score
0,0,0.312384
1,1,0.257904
2,2,0.572171
3,3,0.222919
4,4,0.488934
...,...,...
95,95,0.263370
96,96,0.751800
97,97,0.359785
98,98,0.302357


In [ ]:
output_csv = Path('/content/drive/MyDrive/pls_agent/results/qwen_metrics_align_team.csv')
# Ensure the output directory exists
output_csv.parent.mkdir(parents=True, exist_ok=True)

# Save the results_df to the specified path
results_df.to_csv(output_csv)
print(f'Wrote detailed results to {output_csv}')

Wrote detailed results to /content/drive/MyDrive/pls_agent/results/qwen_metrics_align_team.csv


In [ ]:
results_df.head(50)

,technical_text,reference_summary,generation,provider,bertscore_f1,flesch_reading_ease,flesch_kincaid_grade,gunning_fog,smog_index,coleman_liau_index,align_score,row_id
0,Background\nLumbar spinal stenosis with neurog...,Non‐surgical treatment for spinal stenosis wit...,"In older adults, doctors often diagnose a cond...",qwen+finetuned,0.545525,69.31,8.3,10.19,10.1,11.14,0.312384,0
1,Background\nOlder patients with multiple healt...,Interventions for involving older patients wit...,Old patients with lots of other illnesses like...,qwen+finetuned,0.603298,83.46,4.9,6.22,7.3,8.92,0.257904,1
2,Background\nBeta‐blockers are an essential par...,Beta‐blockers for children with congestive hea...,Children with congested heart failings need sp...,qwen+finetuned,0.596289,74.90,6.1,7.33,8.3,11.64,0.572171,2
3,Background\nThalassaemia is a genetic disorder...,Removal of the spleen in people with thalassae...,People with thalamus major or intermediate typ...,qwen+finetuned,0.577115,82.95,5.1,6.46,7.5,9.10,0.222919,3
4,Background\nApproximately 600 million children...,"One, two or three times a week iron supplement...",Children who are too young to take their iron ...,qwen+finetuned,0.605809,85.83,6.1,8.37,6.9,7.31,0.488934,4
5,Background\nHeart failure (HF) is a chronic di...,mHealth‐delivered education interventions in h...,People with heart failures have a hard time ma...,qwen+finetuned,0.531550,74.69,6.2,7.22,8.1,10.32,0.350424,5
6,Background\nThe method of delivering a diagnos...,Ways of communicating to a woman that she has ...,Women who receive a diagnosis for breast cance...,qwen+finetuned,0.633039,61.56,9.2,10.27,11.0,12.12,0.702952,6
7,Background\nTinnitus affects 10% to 15% of the...,Sound therapy (using amplification devices or ...,About 1 in 6 adults have ringing in their ears...,qwen+finetuned,0.546770,71.65,7.4,7.81,7.9,9.39,0.249036,7
8,Background\nTuberculous pericarditis can impai...,Treatment for tuberculosis infection of the me...,People with tubercular pericarthritis face ris...,qwen+finetuned,0.592019,40.85,10.9,11.89,12.4,16.75,0.542287,8
9,Background\nVitamin D deficiency is common wor...,Vitamin D supplementation for term breastfed i...,Infants need vitamin D to grow strong bones. B...,qwen+finetuned,0.557065,78.14,4.9,4.65,7.9,6.41,0.221113,9


In [ ]:
len(results_df)

100

## 13. Agregación y comparación

Calcula promedios macro de latencia, tokens, BERTScore, AlignScore y legibilidad; AlignScore quedará en NaN si no se corrió el bloque aislado.


In [ ]:
summary = (
    results_df.groupby('provider')[
        [
            'prompt_tokens',
            'completion_tokens',
            'total_tokens',
            'latency_seconds',
            'bertscore_f1',
            'align_score',
            'flesch_reading_ease',
            'flesch_kincaid_grade',
            'gunning_fog',
            'smog_index',
            'coleman_liau_index',
        ]
    ]
    .mean()
    .rename(columns={'latency_seconds': 'avg_latency_seconds'})
)
summary

,bertscore_f1,align_score,flesch_reading_ease,flesch_kincaid_grade,gunning_fog,smog_index,coleman_liau_index
provider,,,,,,,
qwen+finetuned,0.564837,0.362183,73.5226,6.624,7.599,8.604,10.4139


In [ ]:
output_csv = Path('/content/drive/MyDrive/pls_agent/results/predictions_commercial_metrics_complete_team_v2.csv')



# Save the results_df to the specified path
results_df = pd.read_csv(output_csv)
results_df.tail(5)

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,provider,technical_text,reference_summary,prompt,generation,prompt_tokens,completion_tokens,total_tokens,latency_seconds,row_id,bertscore_f1,flesch_reading_ease,flesch_kincaid_grade,gunning_fog,smog_index,coleman_liau_index,align_score
195,195,195,195,claude,Background\nPeople with atrial fibrillation (A...,Atrial fibrillation surgery for patients under...,Background\nPeople with atrial fibrillation (A...,Scientists studied whether people with atrial ...,1690,191,1881,7.565385,195,0.659999,46.10,13.0,13.73,14.3,14.28,0.269048
196,196,196,196,claude,Background\nAsthma is the most common respirat...,Interventions for managing asthma in pregnancy...,Background\nAsthma is the most common respirat...,Asthma is a common breathing problem that affe...,2237,249,2486,8.489365,196,0.643526,39.00,15.8,16.76,14.6,15.73,0.565104
197,197,197,197,claude,Background\nThis review is an update of a prev...,Local anaesthetic sympathetic blockade for com...,Background\nThis review is an update of a prev...,Scientists reviewed studies to see if blocking...,1420,200,1620,7.410084,197,0.664020,56.89,11.0,13.86,13.7,13.12,0.508741
198,198,198,198,claude,Background\nChronic inflammatory demyelinating...,Immune system treatments other than corticoste...,Background\nChronic inflammatory demyelinating...,Chronic inflammatory demyelinating polyradicul...,1979,279,2258,9.524134,198,0.611767,46.20,13.0,14.58,13.6,15.79,0.373927
199,199,199,199,claude,Background\nThis is an updated version of the ...,Antifibrinolytic agents to reduce blood loss i...,Background\nThis is an updated version of the ...,Scientists studied medicines called antifibrin...,1716,264,1980,7.320122,199,0.602977,52.23,12.8,13.85,11.8,14.05,0.377667


## 14. Persistencia de salidas

Guarda la tabla detallada con prompts, respuestas y métricas para reproducir los experimentos fuera del notebook.


In [ ]:
config.output_path.parent.mkdir(parents=True, exist_ok=True)
results_df.to_parquet(config.output_path, index=False)
print(f'Wrote detailed results to {config.output_path}')